# Reference genome data access

This notebook demonstrates the `Ag3` methods for accessing the *Anopheles gambiae* reference genome sequence and its feature (gene/transcript) annotations, including bokeh-based genome track plots.

## Set up the Ag3 data resource

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `contigs`

A property (no arguments) returning a tuple of the reference genome contig names available for *Anopheles gambiae*, i.e. the chromosome arms `2R`, `2L`, `3R`, `3L` and the sex chromosome `X`. These names are the valid values for the `contig`/`region` parameters used throughout the rest of the API (e.g. `genome_sequence`, `genome_features`, `plot_genes`).

In [2]:
ag3.contigs

('2R', '2L', '3R', '3L', 'X')

## `genome_sequence`

Returns a dask array of nucleotide bytes (e.g. `b'a'`, `b'c'`, `b'g'`, `b't'`) giving the reference genome sequence for a requested region.

Parameters:
- `region` (required): a contig name (e.g. `"2L"`, returning the whole chromosome arm), a region string `"{contig}:{start}-{end}"` (1-based, inclusive; commas in the numbers are allowed and stripped), or a gene/transcript feature ID. Here we use a region string to fetch a small 200 bp window rather than a whole chromosome arm, which keeps the output easy to inspect.
- `inline_array` (default `True`): passed through to dask's `from_array()`; controls whether the array is inlined into the dask task graph (can affect performance/graph size for very large arrays, not the result).
- `chunks` (default `"native"`): how the underlying zarr data is chunked for the dask array. `"native"` uses the zarr store's own chunk sizes; other options let you resize chunks (e.g. a memory-size string like `"300 MiB"`, `"auto"`, or a tuple of explicit sizes) — this affects computation/memory characteristics, not the values returned.

In [3]:
seq = ag3.genome_sequence(region="2L:1-200")
seq.compute()

array([b'a', b'a', b'c', b'c', b'a', b't', b'g', b'g', b't', b'c', b'c',
       b'a', b'g', b'a', b'g', b't', b'a', b'c', b'a', b'c', b'a', b't',
       b't', b'g', b'a', b'c', b't', b'a', b't', b'g', b'c', b'a', b'g',
       b'g', b'c', b'c', b't', b'a', b'g', b't', b'a', b'g', b'a', b'c',
       b'g', b'a', b'a', b't', b't', b'c', b't', b'a', b'c', b't', b't',
       b'c', b'c', b't', b't', b'g', b't', b'a', b'a', b't', b'c', b'g',
       b't', b'g', b'g', b'a', b't', b'c', b'c', b'a', b'c', b'a', b'c',
       b't', b'c', b'g', b'a', b'a', b'a', b't', b'g', b'g', b'c', b'c',
       b'g', b'g', b'a', b'a', b'g', b't', b't', b't', b'a', b'c', b'g',
       b'c', b't', b'a', b'c', b'c', b'a', b'g', b'a', b't', b'c', b'a',
       b'a', b't', b'a', b'a', b'c', b't', b'g', b'c', b'g', b'a', b'g',
       b'a', b'a', b'c', b'a', b'a', b'c', b'a', b'a', b't', b'a', b'a',
       b'g', b'a', b'a', b't', b't', b't', b't', b'g', b'a', b'a', b'a',
       b'c', b'a', b'a', b'a', b't', b't', b't', b'

## `genome_features`

Returns a pandas dataframe of genome feature annotations (genes, transcripts, exons, UTRs, CDSs, etc.) in GFF3-like format, with columns `contig`, `source`, `type`, `start`, `end`, `score`, `strand`, `phase`, plus any requested attribute columns.

Parameters:
- `region` (optional): restrict features to a contig, region string, or gene/transcript ID (or a list of these). `None` (the default) returns features genome-wide. Below we use the gene ID `"AGAP004707"` (the voltage-gated sodium channel gene, also known by the name `"para"`), which returns just the gene and all its child features (transcripts, exons, UTRs, CDSs).
- `attributes` (default: the dataset's default set, here `("ID", "Parent", "Name", "description")`): which GFF3 attribute keys to unpack into their own dataframe columns. Pass `"*"` to unpack every attribute present, a single attribute name, or a sequence of names, or `None` to unpack none of them.

In [4]:
df_features = ag3.genome_features(region="AGAP004707")
df_features

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.08)

Load genome features: ⠙ (0:00:00.08)

Load genome features: ⠹ (0:00:00.17)

...

Load genome features: ⠼ (0:00:00.45)

Load genome features: ⠴ (0:00:00.54)

Load genome features: ⠦ (0:00:00.62)

Load genome features: ⠧ (0:00:00.79)

,contig,source,type,start,end,score,strand,phase,ID,Parent,Name,description
0,2L,VectorBase,chromosome,1,49364325,NaN,NaN,NaN,2L,NaN,NaN,NaN
1,2L,VectorBase,gene,2358158,2431617,NaN,+,NaN,AGAP004707,NaN,para,voltage-gated sodium channel [Source:VB Commun...
2,2L,VectorBase,mRNA,2358158,2431617,NaN,+,NaN,AGAP004707-RA,AGAP004707,NaN,NaN
3,2L,VectorBase,exon,2358158,2358304,NaN,+,NaN,NaN,AGAP004707-RA,AGAP004707-RA-E1,NaN
4,2L,VectorBase,CDS,2358158,2358304,NaN,+,0.0,AGAP004707-PA,AGAP004707-RA,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
808,2L,VectorBase,CDS,2430601,2431617,NaN,+,0.0,AGAP004707-PK,AGAP004707-RK,NaN,NaN
809,2L,VectorBase,exon,2430601,2431617,NaN,+,NaN,NaN,AGAP004707-RL,AGAP004707-RA-E30,NaN
810,2L,VectorBase,CDS,2430601,2431617,NaN,+,0.0,AGAP004707-PL,AGAP004707-RL,NaN,NaN
811,2L,VectorBase,exon,2430601,2431617,NaN,+,NaN,NaN,AGAP004707-RM,AGAP004707-RA-E30,NaN


## `plot_transcript`

Plots a single transcript's structure (exons as boxes, introns as chevrons, UTRs and CDS colour-coded) as a bokeh figure, using the genome coordinate axis.

Parameters:
- `transcript` (required): the transcript ID to plot, e.g. one of the `mRNA`-type feature IDs seen in `genome_features` above (`AGAP004707-RA`, `AGAP004707-RB`, ...). Changing this plots a different transcript.
- `sizing_mode` (default `"stretch_width"`): bokeh sizing mode controlling how the figure resizes within its container.
- `width` (default `None`): plot width in pixels; `None` lets `sizing_mode` control the width.
- `height` (default `100`): plot height in pixels.
- `show` (default `True`): if `True`, display the plot immediately (and return `None`); if `False`, return the bokeh figure object without displaying it, useful for combining with other tracks.
- `x_range` (default `None`): a bokeh `Range1d` to link the x-axis with another track/plot; if `None`, a range is computed automatically from the transcript's extent plus a 2 kb margin.
- `toolbar_location` (default `"above"`): where the bokeh toolbar is placed (`"above"`, `"below"`, `"left"`, `"right"`).
- `title` (default `True`): plot title; `True` auto-generates one from the transcript ID and strand, a string sets a custom title, `False`/empty disables it.

**Diagram opportunity:** a labelled illustration of a transcript track (showing how exon boxes, intron chevrons, 5'/3' UTR colouring and CDS colouring map onto the underlying GFF3 feature types) would make the bokeh plot's visual encoding easier to read at a glance.

In [5]:
transcript_id = df_features.query("type == 'mRNA'")["ID"].iloc[0]
print("Plotting transcript:", transcript_id)
ag3.plot_transcript(transcript=transcript_id, height=150)

Plotting transcript: AGAP004707-RA


Load genome features: ⠋ (0:00:00.00)

figure(id='p1011', ...)

## `plot_genes`

Plots a genes track (each gene as a rectangle, above the axis for `+` strand and below for `-` strand) for a genome region, as a bokeh figure. Clicking a gene opens its VectorBase record.

Parameters:
- `region` (required): a contig name or region string to plot genes within. Here we use `"3L:15,000,000-16,000,000"`, a 1 Mbp window, to show several genes at once.
- `sizing_mode` (default `"stretch_width"`): bokeh sizing mode, as above.
- `width` (default `None`): plot width in pixels.
- `height` (default `120`): plot height in pixels.
- `show` (default `True`): show immediately vs. return the figure.
- `toolbar_location` (default `"above"`): bokeh toolbar placement.
- `x_range` (default `None`): link the x-axis to another track; auto-computed from the region if not given.
- `title` (default `None`): plot title.
- `output_backend` (default `"webgl"`): bokeh rendering backend (`"canvas"`, `"webgl"`, `"svg"`); `"webgl"` handles plots with many elements (like dense gene tracks) faster.
- `gene_labels` (default `None`): a mapping of gene ID to custom label text to display above/below each gene rectangle; if `None`, no labels are drawn (useful for highlighting specific genes of interest in a busy region).
- `gene_labelset` (default `None`): an already-constructed bokeh `LabelSet` to add to the figure, for advanced custom labelling beyond `gene_labels`.

**Diagram opportunity:** a small annotated key explaining the `+`/`-` strand rectangle placement (above vs. below the axis) and the tap-to-open-VectorBase interaction would help first-time readers of this track.

In [6]:
ag3.plot_genes(region="3L:15,000,000-16,000,000", gene_labels={"AGAP004707": "para"})

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.09)

Load genome features: ⠹ (0:00:00.17)

Load genome features: ⠸ (0:00:00.33)

figure(id='p1518', ...)

## `canonical_transcript`

Given a gene identifier, returns the transcript ID of that gene's canonical transcript, defined as the transcript with the greatest total exon (transcribed) length.

Parameters:
- `gene` (required): a gene ID (e.g. `"AGAP004707"`) or gene name (matched case-insensitively, e.g. `"para"` — note that despite older documentation, `AGAP004707`'s gene name in the current annotation is `"para"`, not `"Pvr"`). An unrecognised or ambiguous name raises a `ValueError`.

In [7]:
canonical_by_id = ag3.canonical_transcript(gene="AGAP004707")
canonical_by_name = ag3.canonical_transcript(gene="para")
canonical_by_id, canonical_by_name

Load gene data: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

Load gene data: ⠙ (0:00:00.09)

Load gene data: ⠹ (0:00:00.18)

Load genome features: ⠹ (0:00:00.18)

...

Load genome features: ⠸ (0:00:00.39)

Load gene data: ⠼ (0:00:00.45)

Load gene data: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

('AGAP004707-RH', 'AGAP004707-RH')